# 04 — Normalize CHILDES, Benchmark Qwen2.5-1.5B, and Train LoRA

This notebook runs the first controlled model experiment for the Gleason child-language project.

**Experiment:** unchanged base model → supervised LoRA → identical frozen test benchmark.

The frozen benchmark contains **677 test examples**. The preprocessing manifest specifies a child-level
split, up to six context turns, exclusion of `<SILENCE>` from Language SFT, and retention of silence
only in the separate interaction dataset.

## Experimental controls

- Base model: `Qwen/Qwen2.5-1.5B-Instruct`
- Fine-tuning method: PEFT LoRA
- Training: Language SFT train only
- Model selection / early stopping: validation only
- Final comparison: the same frozen test examples before and after LoRA
- Same tokenizer, prompt template, and decoding configuration for Base and LoRA
- Raw CHILDES references are retained; a normalized version is used for the first language experiment.
- No DPO, GRPO, QLoRA, or `<SILENCE>` learning in this experiment.

> **GPU recommended.** A 1.5B model is impractically slow to train on CPU.


In [ ]:
# Run once in a fresh Colab/Jupyter environment if packages are missing.
# !pip install -q -U "transformers>=4.45" "datasets>=2.20" "peft>=0.13" "accelerate>=0.34" sentencepiece pandas matplotlib

from pathlib import Path
import json, re, math, hashlib, random, time
from collections import Counter

import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed,
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "data"
DATA_DIR = DATA_ROOT / "processed" / "sft_baseline"
TRAIN_PATH = DATA_DIR / "language_sft_train.jsonl"
VAL_PATH = DATA_DIR / "language_sft_validation.jsonl"
TEST_PATH = DATA_DIR / "language_sft_test.jsonl"
FROZEN_PATH = DATA_DIR / "gleason_frozen_baseline_test.jsonl"
EXPECTED_SHA_PATH = DATA_DIR / "gleason_frozen_baseline_test.sha256"

OUT = DATA_ROOT / "experiments" / "qwen_lora"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
MAX_LENGTH = 512

# Fixed decoding configuration for BOTH base and LoRA.
# Greedy decoding removes sampling variance from the first controlled comparison.
GENERATION = {
    "do_sample": False,
    "max_new_tokens": 48,
}

set_seed(SEED)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 1. Verify the frozen benchmark

The SHA-256 check protects against accidental changes to the test set between experiments.


In [ ]:
required = [TRAIN_PATH, VAL_PATH, TEST_PATH, FROZEN_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing input files: " + ", ".join(missing))

actual_sha = hashlib.sha256(FROZEN_PATH.read_bytes()).hexdigest()

if EXPECTED_SHA_PATH.exists():
    expected_sha = EXPECTED_SHA_PATH.read_text(encoding="utf-8").strip().split()[0]
    assert actual_sha == expected_sha, (
        f"Frozen benchmark checksum mismatch!\nExpected: {expected_sha}\nActual:   {actual_sha}"
    )

print("Frozen benchmark SHA-256:", actual_sha)
print("PASS: frozen benchmark integrity verified.")


In [ ]:
def read_jsonl(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_raw = read_jsonl(TRAIN_PATH)
val_raw = read_jsonl(VAL_PATH)
test_raw = read_jsonl(TEST_PATH)
frozen_raw = read_jsonl(FROZEN_PATH)

print("Train:", len(train_raw))
print("Validation:", len(val_raw))
print("Test:", len(test_raw))
print("Frozen benchmark:", len(frozen_raw))

assert len(frozen_raw) == len(test_raw), (
    "Frozen and ordinary test files have different sizes. Inspect before proceeding."
)


## 2. Reproducible CHILDES normalization

The original transcription is **never overwritten**. Every example receives both `raw_*` and
`normalized_*` fields.

This first-pass normalizer removes transcription-control markup that should not be emitted literally
by a simulated child while preserving lexical child speech as much as possible.

It handles common CHAT material including:

- media bullets / timing markers;
- retracing and repetition codes such as `[//]` and `[/]`;
- bracketed annotation codes;
- pause-duration notation such as `(7.)` or `(.)`;
- `&=event` / `&-fragment` control forms;
- CHAT terminator artifacts;
- whitespace introduced by removing markup.

`xxx`, `yyy`, and `www` are mapped to `<UNINTELLIGIBLE>` rather than silently deleted.

**Important:** CHAT is a rich transcription format. This is intentionally a conservative modeling
normalizer, not a replacement for a full CHAT parser. The audit table below must be reviewed before
training.


In [ ]:
MEDIA_RE = re.compile(r"\x15\d+_\d+\x15")
BRACKET_CODE_RE = re.compile(r"\[[^\]]+\]")
PAUSE_RE = re.compile(r"\(\s*(?:\d+(?:\.\d*)?|\.)\s*\)")
EVENT_RE = re.compile(r"&=[^\s]+")
FRAGMENT_RE = re.compile(r"&-[^\s]+")
ANGLE_SCOPE_RE = re.compile(r"[<>]")
SPACE_RE = re.compile(r"\s+")

def normalize_childes(text):
    if text is None:
        return ""
    s = str(text)

    # Preserve unintelligibility semantically.
    s = re.sub(r"(?<!\\w)(?:xxx|yyy|www)(?!\\w)", " <UNINTELLIGIBLE> ", s, flags=re.I)

    # Remove timing and transcription-control markup.
    s = MEDIA_RE.sub(" ", s)
    s = BRACKET_CODE_RE.sub(" ", s)
    s = PAUSE_RE.sub(" ", s)
    s = EVENT_RE.sub(" ", s)
    s = FRAGMENT_RE.sub(" ", s)
    s = ANGLE_SCOPE_RE.sub("", s)

    # CHAT terminators often occur as separated punctuation tokens.
    s = re.sub(r"\s+[.!?](?=\s|$)", " ", s)

    # Clean residual delimiters without rewriting lexical content.
    s = s.replace("‡", " ").replace("„", " ")
    s = SPACE_RE.sub(" ", s).strip()

    return s


audit_examples = [
    "I want xxx .",
    "I [/] I want that .",
    "no [//] yes .",
    "I want (7.) that .",
    "<I want> [/] I want it .",
]

for x in audit_examples:
    print("RAW :", x)
    print("NORM:", normalize_childes(x))
    print()


### Normalize instructions, conversational context, and references


In [ ]:
def normalize_record(rec, frozen=False):
    r = dict(rec)

    # Preserve originals.
    r["raw_input"] = r["input"]
    target_key = "reference" if frozen else "output"
    r["raw_reference"] = r[target_key]

    # Normalize each dialogue line while preserving speaker labels.
    normalized_lines = []
    for line in r["input"].splitlines():
        if ":" in line:
            label, content = line.split(":", 1)
            normalized_lines.append(f"{label}: {normalize_childes(content)}")
        else:
            normalized_lines.append(normalize_childes(line))

    r["normalized_input"] = "\n".join(normalized_lines)
    r["normalized_reference"] = normalize_childes(r[target_key])
    return r

train = [normalize_record(r) for r in train_raw]
val = [normalize_record(r) for r in val_raw]
test = [normalize_record(r) for r in test_raw]
frozen = [normalize_record(r, frozen=True) for r in frozen_raw]

# Empty targets are not useful for Language SFT.
train = [r for r in train if r["normalized_reference"].strip()]
val = [r for r in val if r["normalized_reference"].strip()]

print("After normalization:")
print("Train:", len(train))
print("Validation:", len(val))
print("Frozen test:", len(frozen))


### Audit what normalization changed


In [ ]:
changed = []
for split_name, records in [("train", train), ("validation", val), ("test", frozen)]:
    for r in records:
        if r["raw_reference"] != r["normalized_reference"]:
            changed.append({
                "split": split_name,
                "id": r.get("id"),
                "raw": r["raw_reference"],
                "normalized": r["normalized_reference"],
            })

changed_df = pd.DataFrame(changed)
print(f"References changed by normalization: {len(changed_df):,}")
display(changed_df.head(50))

changed_df.to_csv(OUT / "normalization_audit.csv", index=False)


## 3. Load the exact tokenizer and inspect model-token lengths

This is the token-length check that Notebook 03 intentionally deferred until a base model was chosen.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

def messages_for_record(r, include_answer=False):
    messages = [
        {"role": "system", "content": r["instruction"]},
        {"role": "user", "content": r["normalized_input"]},
    ]
    if include_answer:
        messages.append({"role": "assistant", "content": r["normalized_reference"]})
    return messages

def prompt_text(r):
    return tokenizer.apply_chat_template(
        messages_for_record(r, include_answer=False),
        tokenize=False,
        add_generation_prompt=True,
    )

def full_text(r):
    return tokenizer.apply_chat_template(
        messages_for_record(r, include_answer=True),
        tokenize=False,
        add_generation_prompt=False,
    )

length_rows = []
for split_name, records in [("train", train), ("validation", val), ("test", frozen)]:
    for r in records:
        p = tokenizer(prompt_text(r), add_special_tokens=False)["input_ids"]
        full = tokenizer(full_text(r), add_special_tokens=False)["input_ids"]
        length_rows.append({
            "split": split_name,
            "id": r.get("id"),
            "prompt_tokens": len(p),
            "total_tokens": len(full),
            "target_tokens": max(0, len(full) - len(p)),
        })

length_df = pd.DataFrame(length_rows)
display(
    length_df.groupby("split")[["prompt_tokens", "target_tokens", "total_tokens"]]
    .agg(["mean", "median", "max"])
    .round(1)
)

over = (length_df["total_tokens"] > MAX_LENGTH).mean()
print(f"Fraction over MAX_LENGTH={MAX_LENGTH}: {over:.2%}")


In [ ]:
plt.figure(figsize=(8, 5))
for split, g in length_df.groupby("split"):
    plt.hist(g["total_tokens"], bins=35, alpha=.5, label=split)
plt.axvline(MAX_LENGTH, linestyle="--")
plt.xlabel("Qwen tokenizer tokens")
plt.ylabel("Examples")
plt.title("Full SFT sequence lengths")
plt.legend()
plt.tight_layout()
plt.show()


## 4. Tokenize SFT data with **prompt masking**

Only assistant-response tokens contribute to training loss. System instructions and caregiver/history
tokens receive label `-100`.

This matters: otherwise the model is partly trained to reproduce the prompt instead of specifically
learning the child response.


In [ ]:
def tokenize_sft_record(r):
    prompt = prompt_text(r)
    full = full_text(r)

    p_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )["input_ids"]

    enc = tokenizer(
        full,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = enc["input_ids"]
    attention_mask = enc["attention_mask"]
    labels = input_ids.copy()

    prompt_len = min(len(p_ids), len(labels))
    labels[:prompt_len] = [-100] * prompt_len

    # Drop examples where truncation removed the entire answer.
    has_target = any(x != -100 for x in labels)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "has_target": has_target,
    }

def make_hf_dataset(records):
    ds = Dataset.from_list(records)
    ds = ds.map(tokenize_sft_record)
    ds = ds.filter(lambda x: x["has_target"])
    keep = {"input_ids", "attention_mask", "labels"}
    remove = [c for c in ds.column_names if c not in keep]
    return ds.remove_columns(remove)

train_ds = make_hf_dataset(train)
val_ds = make_hf_dataset(val)

print(train_ds)
print(val_ds)


## 5. Fixed benchmark-generation function

This function is used **unchanged** for the base model and LoRA adapter.

Greedy decoding is the default for this first controlled comparison. Later experiments can separately
study stochastic child-language generation.


In [ ]:
def load_base_model():
    dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
        torch.float16 if torch.cuda.is_available() else torch.float32
    )

    kwargs = {"torch_dtype": dtype}
    if torch.cuda.is_available():
        kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **kwargs)
    model.eval()
    return model

@torch.inference_mode()
def run_benchmark(model, records, output_path):
    rows = []
    device = next(model.parameters()).device

    for i, r in enumerate(records):
        prompt = prompt_text(r)
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False,
        ).to(device)

        out = model.generate(
            **inputs,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **GENERATION,
        )

        new_tokens = out[0, inputs["input_ids"].shape[1]:]
        generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

        rows.append({
            "benchmark_id": r["benchmark_id"],
            "id": r.get("id"),
            "child_id": r["child_id"],
            "age_months": r["age_months"],
            "condition": r.get("condition"),
            "raw_reference": r["raw_reference"],
            "reference": r["normalized_reference"],
            "generation": generated,
        })

        if (i + 1) % 50 == 0 or i + 1 == len(records):
            print(f"{i+1}/{len(records)}")

    pd.DataFrame(rows).to_json(
        output_path, orient="records", lines=True, force_ascii=False
    )
    return pd.DataFrame(rows)


## 6. Run and save the **untouched base-model benchmark**

Run this cell **before training**. It may take a while for all 677 examples.

The result is your immutable pre-LoRA baseline.


In [ ]:
base_generations_path = OUT / "base_frozen_generations.jsonl"
if base_generations_path.exists():
    base_generations = pd.read_json(base_generations_path, orient="records", lines=True)
    expected_benchmark_ids = [r["benchmark_id"] for r in frozen]
    if base_generations["benchmark_id"].tolist() != expected_benchmark_ids:
        raise ValueError("Existing baseline generations do not match the frozen benchmark.")
    base_model = None
    print(f"Reusing {len(base_generations):,} verified baseline generations.")
else:
    base_model = load_base_model()
    base_generations = run_benchmark(
        base_model,
        frozen,
        base_generations_path
    )

display(base_generations.head(10))


Free base-model memory before creating the trainable LoRA model.


In [ ]:
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 7. Configure LoRA

For Qwen2.5, LoRA is applied to the major attention and MLP projection layers. The base weights remain
frozen; only low-rank adapter parameters are trained.


In [ ]:
base_for_lora = load_base_model()
base_for_lora.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(base_for_lora, lora_config)
model.print_trainable_parameters()


## 8. Train LoRA

Initial settings are intentionally conservative:

- up to **3 epochs**;
- learning rate `2e-4`;
- effective batch size increased with gradient accumulation;
- validation loss used for model selection;
- early stopping enabled.

Do not change these based on test-set results.


In [ ]:
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

args = TrainingArguments(
    output_dir=str(OUT / "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

train_result = trainer.train()
print(train_result)


### Save the best adapter and training metadata


In [ ]:
ADAPTER_DIR = OUT / "best_lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

training_summary = {
    "base_model": MODEL_ID,
    "seed": SEED,
    "max_length": MAX_LENGTH,
    "generation": GENERATION,
    "lora": {
        "r": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
    },
    "train_examples_after_normalization_and_tokenization": len(train_ds),
    "validation_examples_after_normalization_and_tokenization": len(val_ds),
    "frozen_test_examples": len(frozen),
    "frozen_test_sha256": actual_sha,
    "best_checkpoint": trainer.state.best_model_checkpoint,
    "best_validation_metric": trainer.state.best_metric,
}

with (OUT / "training_summary.json").open("w", encoding="utf-8") as f:
    json.dump(training_summary, f, indent=2)

training_summary


## 9. Run LoRA on the **same frozen benchmark**

No decoding parameter is changed from the base-model run.


In [ ]:
model.eval()

lora_generations = run_benchmark(
    model,
    frozen,
    OUT / "lora_frozen_generations.jsonl"
)

display(lora_generations.head(10))


## 10. Sanity metrics

These are **not** the complete developmental benchmark. They provide immediate checks for response
length, empty generations, unintelligibility tokens, and exact copying.

Notebook 05 should implement the paper-aligned linguistic metrics and dynamic multi-turn evaluation.


In [ ]:
def simple_metrics(df, system_name):
    x = df.copy()
    x["generation_words"] = x["generation"].fillna("").str.split().str.len()
    x["reference_words"] = x["reference"].fillna("").str.split().str.len()
    x["empty"] = x["generation"].fillna("").str.strip().eq("")
    x["unintelligible"] = x["generation"].fillna("").str.contains(
        "<UNINTELLIGIBLE>", regex=False
    )
    x["exact_match"] = (
        x["generation"].fillna("").str.strip().str.lower()
        == x["reference"].fillna("").str.strip().str.lower()
    )

    return {
        "system": system_name,
        "examples": len(x),
        "mean_generation_words": x["generation_words"].mean(),
        "median_generation_words": x["generation_words"].median(),
        "mean_reference_words": x["reference_words"].mean(),
        "empty_rate": x["empty"].mean(),
        "unintelligible_rate": x["unintelligible"].mean(),
        "exact_match_rate": x["exact_match"].mean(),
    }

comparison = pd.DataFrame([
    simple_metrics(base_generations, "Base"),
    simple_metrics(lora_generations, "LoRA"),
])

display(comparison.round(4))
comparison.to_csv(OUT / "base_vs_lora_sanity_metrics.csv", index=False)


### Side-by-side qualitative audit


In [ ]:
side = base_generations[
    ["benchmark_id", "age_months", "condition", "reference", "generation"]
].rename(columns={"generation": "base_generation"}).merge(
    lora_generations[["benchmark_id", "generation"]].rename(
        columns={"generation": "lora_generation"}
    ),
    on="benchmark_id",
    validate="one_to_one",
)

display(side.head(50))
side.to_csv(OUT / "base_vs_lora_generations.csv", index=False)


## 11. Reproducibility checks and final artifact list


In [ ]:
# Benchmark IDs must be identical and in the same order.
assert list(base_generations["benchmark_id"]) == list(lora_generations["benchmark_id"])
assert len(base_generations) == len(frozen)
assert len(lora_generations) == len(frozen)

print("PASS: Base and LoRA used the same frozen benchmark.")
print("PASS: Generation configuration was shared by the benchmark function.")
print("\nArtifacts:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)


# What comes next

Do **not** start DPO/GRPO yet.

The next notebook should evaluate Base and LoRA using the developmentally relevant measurements from
the project benchmark:

- utterance length;
- content-word / lexical density;
- word concreteness where lexicon coverage permits;
- syntactic complexity;
- semantic caregiver→child alignment;
- dialogue-level diversity.

It should then run a **dynamic multi-turn condition** in which a child agent and caregiver agent
generate an interaction from a fixed CHILDES seed, because single-turn performance alone does not
establish that the model remains child-like over sustained conversation.

The raw and normalized references saved here allow the evaluation to distinguish model behavior from
CHAT transcription artifacts.
